# StateGraph: Nodes, Edges, and State

**What you will build:** a graph by hand — the real LangGraph primitive. We'll implement the **routing** pattern from 0.4 as a graph, so you can compare "a few `if`s in Python" with "an explicit state machine" and feel when the graph is worth it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/21_stategraph_fundamentals.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## The three pieces

- **State** — a `TypedDict` that every node reads from and writes to. It's the shared memory of the graph.
- **Nodes** — plain functions `state -> partial state update`.
- **Edges** — how control flows between nodes; **conditional edges** branch based on the state.

We'll build: classify a support ticket, then route to the matching specialist node.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str        # the incoming ticket
    category: str    # filled by the classifier
    reply: str       # filled by a specialist

def classify(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "Classify as ONE word: billing, technical, or other."},
                    {"role": "user", "content": state["text"]}])
    return {"category": r.content.strip().lower()}

def billing(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "You are a billing specialist. Be precise."},
                    {"role": "user", "content": state["text"]}])
    return {"reply": r.content}

def technical(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "You are a technical support engineer. Give steps."},
                    {"role": "user", "content": state["text"]}])
    return {"reply": r.content}

def other(state: State) -> dict:
    return {"reply": "Forwarding your message to a human agent."}

## Wire the nodes with edges

`add_conditional_edges` is the branch: a router function returns the name of the next node.

In [ ]:
def route(state: State) -> str:
    c = state["category"]
    if "billing" in c:   return "billing"
    if "technical" in c: return "technical"
    return "other"

builder = StateGraph(State)
builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("technical", technical)
builder.add_node("other", other)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route,
                              {"billing": "billing", "technical": "technical", "other": "other"})
for n in ("billing", "technical", "other"):
    builder.add_edge(n, END)

graph = builder.compile()

```{note}
The router uses `"billing" in c` on purpose-built leniency — the classifier node may answer "Billing." or "billing issue" and we still route it. You know from 0.4 that the *robust* version is typed output, and LangChain has the equivalent of 1.3's `output_type`: `llm.with_structured_output(Route)` with a `Literal` field. We keep the plain version here to stay focused on graph mechanics; in anything real, type the classifier.
```

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))     # PNG via mermaid.ink (needs internet)
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())                  # pure text (Mermaid DSL), no extra deps

Run it — the state flows in, gets a `category`, branches, and comes out with a `reply`:

In [ ]:
out = graph.invoke({"text": "I was charged twice for my subscription this month."})
print("category:", out["category"])
print("reply:", out["reply"][:200], "...")

You built the same routing logic as 0.4 — but now it's an explicit, drawable object, and the `State` carries data cleanly between steps. For three branches that's arguably heavier than plain `if`s. The graph pays off when the flow has loops, parallel paths, human pauses, or many nodes — everything coming in 2.3–2.6.

## Parallel branches — and why reducers exist

Routing picks *one* path. Graphs can also take **several at once**: give a node multiple outgoing edges and LangGraph runs the targets concurrently in the same step. Which immediately raises the question the dropdown below formalizes — if two nodes write to the state at the same time, *who wins?* The answer you want is usually "nobody — combine them", and that's a **reducer**:

In [ ]:
from typing import Annotated
import operator

class Analysis(TypedDict):
    text: str
    findings: Annotated[list[str], operator.add]   # concurrent writes APPEND instead of clobbering

def tone(state: Analysis) -> dict:
    r = llm.invoke([{"role": "system", "content": "In 5 words: what's the emotional tone?"},
                    {"role": "user", "content": state["text"]}])
    return {"findings": [f"tone: {r.content}"]}

def urgency(state: Analysis) -> dict:
    r = llm.invoke([{"role": "system", "content": "One word: low, medium or high urgency?"},
                    {"role": "user", "content": state["text"]}])
    return {"findings": [f"urgency: {r.content}"]}

def language(state: Analysis) -> dict:
    r = llm.invoke([{"role": "system", "content": "One word: what language is this?"},
                    {"role": "user", "content": state["text"]}])
    return {"findings": [f"language: {r.content}"]}

pb = StateGraph(Analysis)
for name, fn in [("tone", tone), ("urgency", urgency), ("language", language)]:
    pb.add_node(name, fn)
    pb.add_edge(START, name)      # three edges out of START = three branches, run concurrently
    pb.add_edge(name, END)

par = pb.compile()
out = par.invoke({"text": "¡Es la tercera vez que os escribo y sigo sin poder entrar en mi cuenta!"})
for f in out["findings"]:
    print("-", f)

Three nodes ran in the same step, and `findings` holds all three results — because `operator.add` told LangGraph to *concatenate* concurrent writes. Delete the `Annotated[..., operator.add]` and the same graph **refuses to run** (`InvalidUpdateError`: the key received multiple values in one step). That error is a feature: 0.4's `asyncio.gather` version of this would have silently kept whichever result you wrote last; the graph makes the merge decision explicit and mandatory. This is also pattern 3 of 0.4 (parallelization) as topology instead of code.

### Fan-out when you don't know the width — `Send`

The parallel graph above had **exactly three** branches, wired at build time. But often the width is only known at *runtime* — one worker per ticket in a batch, per document in an upload, per section of a report. You can't draw those edges in advance, because you don't yet know how many there are.

`Send` is the answer: from a conditional edge, return a **list of `Send(node, state)`** objects — one per item — and LangGraph launches that many copies of the worker node in the same step. Their results merge through the same `operator.add` reducer you just used.

In [ ]:
from langgraph.types import Send

class Batch(TypedDict):
    tickets: list[str]                              # arrives at runtime — width unknown
    summaries: Annotated[list[str], operator.add]   # workers append concurrently

def summarize_one(state: dict) -> dict:             # runs once per ticket
    r = llm.invoke([{"role": "system", "content": "Summarize this ticket in 8 words or fewer."},
                    {"role": "user", "content": state["ticket"]}])
    return {"summaries": [r.content.strip()]}

def fan_out(state: Batch):                          # a conditional edge, not a node
    return [Send("summarize_one", {"ticket": t}) for t in state["tickets"]]

fb = StateGraph(Batch)
fb.add_node("summarize_one", summarize_one)
fb.add_conditional_edges(START, fan_out, ["summarize_one"])   # START -> N workers, count from the data
fb.add_edge("summarize_one", END)
fanned = fb.compile()

out = fanned.invoke({"tickets": ["I was double-charged on my card.",
                                 "The app won't open after the update.",
                                 "How do I change my email address?"]})
for s in out["summaries"]:
    print("-", s)

No edges were drawn to a fixed set of nodes — `fan_out` created the work at runtime. This is 0.4's parallelization pattern again, but **data-driven**: the static fan-out needs you to know the branches in advance; `Send` builds them from the input. Same reducer, same merge — only the *count* moved from build time to run time.

::::{dropdown} 🔧 Under the hood: state updates and the "merge problem"
:color: info

Each node returns a **partial** update (`{"category": ...}`) that LangGraph merges into the state. By default a key is *overwritten*. If two nodes write the same key (e.g. in parallel branches), the last write wins and you silently lose data. To *accumulate* instead, annotate the field with a reducer:

```python
from typing import Annotated
import operator
class State(TypedDict):
    findings: Annotated[list, operator.add]   # appends instead of overwriting
```

This is the same idea behind `add_messages`, the reducer that makes a `messages` list grow correctly — which we use in the next notebook.

One more routing tool for your map: a node can return `Command(goto="billing", update={...})` — state update and routing decision in one object, instead of a separate router function. You'll meet it in 2.6, where agents hand control to each other.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a route.** Give the ticket graph a `refund` category with its own specialist node. **Done when:** a refund ticket comes out with the new reply — and you needed edits in exactly three places (classifier prompt, a node, the routing map).
2. **Break the merge on purpose.** Remove the `Annotated[..., operator.add]` from `findings` and re-run the parallel graph. **Done when:** you've seen the `InvalidUpdateError` and can say why the sequential ticket graph never needs a reducer.
3. **Draw before you run.** Sketch (on paper) what `par.get_graph().draw_mermaid()` will output, then print it. **Done when:** your sketch matches — fan-out, no router.
4. **Grow the batch.** Pass five tickets to the `Send` graph instead of three, changing nothing else. **Done when:** five summaries come back and you can explain why no code or edge changed — the width lives in the data.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 —**

```python
def refund(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "You are a refunds specialist. Explain the steps and timeline."},
                    {"role": "user", "content": state["text"]}])
    return {"reply": r.content}

# classifier prompt: "... ONE word: billing, technical, refund, or other."
# builder: add_node("refund", refund); add "refund": "refund" to the routing map; add_edge("refund", END)
```

Three touch points, all declarative. Compare with 0.4's version where the same change was one `elif` — the graph is *more* ceremony for simple flows (2.1's honest admission) and pays off when the structure grows.

**2 —** The error names the key and the step. The sequential graph never triggers it because only one node runs per step — reducers exist for *concurrent* writers, and `category`/`reply` each have exactly one writer anyway. Rule of thumb: any state key that parallel branches touch needs a reducer; keys with a single writer don't.

**4 —** `fanned.invoke({"tickets": [ ...five strings... ]})` and nothing else. The graph never names a count; `fan_out` emits one `Send` per item, so the worker count follows the list length. That is exactly what `Send` buys over the static fan-out: the topology is drawn from the data, not the code.
::::

**What's next:** in **2.2** we build the **agent loop itself as a graph** — the ReAct cycle from 0.3, drawn with a real cycle in it.